# End-to-end AutoGluon run (Experiment 4)

Fits the 35 gold targets with AutoGluon forced into each applicable task (280 fits, folds 3–4). Every fit is saved to Drive as soon as it finishes. After a disconnect, run cells 1–3 again and then cell 5: finished fits are skipped.

**Runtime:** CPU (the default) is fine. About 2.5–3.5 h for cell 5.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/task_inference_e2e'   # change if you uploaded elsewhere
import os
for f in ('config.json', 'targets.csv', 'e2e_lib.py', 'data'):
    assert os.path.exists(os.path.join(BASE, f)), f'missing {f}'
print('ok')

## 2. Install AutoGluon (~3–5 min)

If Colab asks you to restart the session after this, click **Restart**, then run cell 1 and continue with cell 3 (do not reinstall).

In [ ]:
%pip install -q "autogluon.tabular[all]"

## 3. Load the library

In [ ]:
import sys, importlib
sys.path.insert(0, BASE)
import e2e_lib
importlib.reload(e2e_lib)
cfg, targets = e2e_lib.load_bundle(BASE)
print(len(targets), 'targets;', cfg)

## 4. Smoke test (~2 min, small table, writes to /content)

Checks the two non-default configurations (Poisson count regression, multiclass with quadratic kappa). Both lines should say `ok`.

In [ ]:
e2e_lib.run(BASE, '/content/smoke_e2e', only={'I002'}, tasks=['COUNT_REGRESSION', 'ORDINAL'], folds=[3], time_limit=20)
_ = e2e_lib.summarize('/content/smoke_e2e')

## 5. Full run (~2.5–3.5 h, resumable)

In [ ]:
e2e_lib.run(BASE, BASE + '/results')

## 6. Summary and zip

Download `e2e_results.zip` from the Drive folder and unzip it into `results/downstream/autogluon_fits/` in the project.

In [ ]:
import shutil
_ = e2e_lib.summarize(BASE + '/results')
shutil.make_archive(BASE + '/e2e_results', 'zip', BASE + '/results')
print('wrote', BASE + '/e2e_results.zip')

### Optional: retry fits that failed

Only if cell 6 lists errors that look like a disconnect or out-of-memory, not a real pipeline failure.

In [ ]:
e2e_lib.run(BASE, BASE + '/results', retry_errors=True)